In [0]:
from pyspark.sql import functions as F

df = spark.table("insureflow.bronze.policy_raw")

display(df)

In [0]:
failed_df = df.filter(F.col('policy_id').isNull())
failed_count = failed_df.count()

In [0]:
total_count = df.count()

status = 'PAss' if failed_count ==0 else 'FAIL'

print(f"total_records: {total_count}")
print(f"Failed Records: {failed_count}")
print(f"Status        : {status}")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
schema = StructType([
    StructField("policy_id", StringType(), True)
])

In [0]:
bad_record = df.union(
            spark.createDataFrame(
                    [(None,)],
                    schema
            )
)

test_df = df.unionByName(bad_record)

In [0]:
from pyspark.sql import functions as F

test_df = (
    df.withColumn(
        "policy_id",
        F.when(F.monotonically_increasing_id() == 0, None)
         .otherwise(F.col("policy_id"))
    )
)

In [0]:
failed_df = test_df.filter(F.col('policy_id').isNull())
failed_count = failed_df.count()

total_count = test_df.count()

status = 'PAss' if failed_count ==0 else 'FAIL'

print(f"total_records: {total_count}")
print(f"Failed Records: {failed_count}")
print(f"Status        : {status}")

In [0]:
duplicate_df = (
                df.groupBy(F.col('policy_id')).count().filter(F.col('count')>1)
)
display(duplicate_df)

In [0]:
duplicate_count = duplicate_df.count()
print("Duplicate policy IDs:", duplicate_count)

In [0]:
test_duplicate_df = df.unionByName(
    df.limit(1)
)

In [0]:
duplicate_df = (
    test_duplicate_df
    .groupBy("policy_id")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_df.count()
status = 'Pass' if duplicate_count ==0 else 'Fail'

print("Total Records      :", test_duplicate_df.count())
print("Duplicate Policy IDs:", duplicate_count)
print(status)

In [0]:
invalid_premium_df = df.filter(F.col('premium')<0)
failed_count = invalid_premium_df.count()
total_count = df.count()
status = 'pass' if failed_count==0 else 'fail'

print("Total Records :", total_count)
print("Failed Records:", failed_count)
print("Status        :", status)

In [0]:
test_premium_df = (
    df.withColumn(
        "premium",
        F.when(
            F.monotonically_increasing_id() == 0,
            F.lit(-500)
        ).otherwise(F.col("premium"))
    )
)

In [0]:
invalid_premium_df = test_premium_df.filter(
    F.col("premium") < 0
)

failed_count = invalid_premium_df.count()
total_count = test_premium_df.count()

status = "PASS" if failed_count == 0 else "FAIL"

print("Total Records :", total_count)
print("Failed Records:", failed_count)
print("Status        :", status)

In [0]:
df.select('status').distinct().show()

In [0]:
allowed_status = ['Pending', 'Expired', 'Active', 'Cancelled']

invalid_status_df = df.filter(
                                ~F.col('status').isin(allowed_status)
)

failed_count = invalid_status_df.count()
total_count = df.count()

status = 'pass' if failed_count ==0 else 'fail'

print("Total Records :", total_count)
print("Failed Records:", failed_count)
print("Status        :", status)

In [0]:
test_status_df = (
    df.withColumn(
        "status",
        F.when(
            F.monotonically_increasing_id() == 0,
            F.lit("Unknown")
        ).otherwise(F.col("status"))
    )
)

In [0]:
invalid_status_df = test_status_df.filter(
    ~F.col("status").isin(allowed_status)
)

failed_count = invalid_status_df.count()
total_count = test_status_df.count()

status = "PASS" if failed_count == 0 else "FAIL"

print("Total Records :", total_count)
print("Failed Records:", failed_count)
print("Status        :", status)

In [0]:
df.printSchema()

In [0]:
display(
    df.select(
        "effective_date",
        "expiry_date"
    ).limit(10)
)

In [0]:
invalid_date_df = df.filter(
                        F.col('effective_date')> F.col('expiry_date')
)

failed_count = invalid_date_df.count()
total_count = df.count()

status = "PASS" if failed_count == 0 else "FAIL"

print("Total Records :", total_count)
print("Failed Records:", failed_count)
print("Status        :", status)

In [0]:
display(
    invalid_date_df.select(
        "policy_id",
        "effective_date",
        "expiry_date"
    )
)

In [0]:
test_date_df = (
    df.withColumn(
        "expiry_date",
        F.when(
            F.monotonically_increasing_id() == 0,
            F.to_date(F.lit("2025-01-01"))
        ).otherwise(F.col("expiry_date"))
    )
)

In [0]:
invalid_date_df = test_date_df.filter(
    F.col("effective_date") > F.col("expiry_date")
)

failed_count = invalid_date_df.count()
total_count = test_date_df.count()

status = "PASS" if failed_count == 0 else "FAIL"

print("Total Records :", total_count)
print("Failed Records:", failed_count)
print("Status        :", status)

In [0]:
def create_dq_result(
    rule_id,
    rule_type,
    rule_description,
    total_records,
    failed_records,
    severity
):
    
    passed_records = total_records - failed_records
    
    status = (
        "PASS"
        if failed_records == 0
        else "FAIL"
    )
    
    return {
        "rule_id": rule_id,
        "rule_type": rule_type,
        "rule_description": rule_description,
        "total_records": total_records,
        "failed_records": failed_records,
        "passed_records": passed_records,
        "status": status,
        "severity": severity
    }

In [0]:
dq_results = []

total_records = df.count()

In [0]:
failed = df.filter(
    F.col("policy_id").isNull()
).count()

dq_results.append(
    create_dq_result(
        rule_id="DQ001",
        rule_type="NOT_NULL",
        rule_description="policy_id must not be NULL",
        total_records=total_records,
        failed_records=failed,
        severity="CRITICAL"
    )
)

In [0]:
failed = (
    df.groupBy("policy_id")
      .count()
      .filter(F.col("count") > 1)
      .count()
)

dq_results.append(
    create_dq_result(
        rule_id="DQ002",
        rule_type="DUPLICATE",
        rule_description="policy_id must be unique",
        total_records=total_records,
        failed_records=failed,
        severity="CRITICAL"
    )
)

In [0]:
failed = df.filter(
    F.col("premium") < 0
).count()

dq_results.append(
    create_dq_result(
        rule_id="DQ003",
        rule_type="RANGE",
        rule_description="premium must be greater than or equal to zero",
        total_records=total_records,
        failed_records=failed,
        severity="HIGH"
    )
)

In [0]:
allowed_status = [
    "Pending",
    "Expired",
    "Active",
    "Cancelled"
]

failed = df.filter(
    ~F.col("status").isin(allowed_status)
).count()

dq_results.append(
    create_dq_result(
        rule_id="DQ004",
        rule_type="ALLOWED_VALUE",
        rule_description="status must contain an allowed value",
        total_records=total_records,
        failed_records=failed,
        severity="HIGH"
    )
)

In [0]:
failed = df.filter(
    F.col("effective_date") > F.col("expiry_date")
).count()

dq_results.append(
    create_dq_result(
        rule_id="DQ005",
        rule_type="DATE_ORDER",
        rule_description="effective_date must be less than or equal to expiry_date",
        total_records=total_records,
        failed_records=failed,
        severity="HIGH"
    )
)

In [0]:
dq_result_df = spark.createDataFrame(dq_results)

display(dq_result_df)

In [0]:
import uuid

run_id = str(uuid.uuid4())
print(run_id)

In [0]:
from pyspark.sql import functions as F

dq_result_df = (
    dq_result_df
    .withColumn("run_id", F.lit(run_id))
    .withColumn("table_name", F.lit("policy"))
    .withColumn("execution_timestamp", F.current_timestamp())
)

In [0]:
display(dq_result_df)

In [0]:
dq_result_df.printSchema()

In [0]:
(dq_result_df.write 
        .format("delta") 
        .mode("overwrite") 
        .saveAsTable("insureflow.audit.dq_results")
)

In [0]:
display(
    spark.table("insureflow.audit.dq_results")
)

In [0]:
spark.sql("""
DESCRIBE DETAIL insureflow.audit.dq_results
""").show(truncate=False)

In [0]:
import uuid

run_id_2 = str(uuid.uuid4())

print(run_id_2)

In [0]:
dq_result_df_2 = (
    dq_result_df
    .withColumn("run_id", F.lit(run_id_2))
    .withColumn("execution_timestamp", F.current_timestamp())
)

In [0]:
dq_result_df_2.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("insureflow.audit.dq_results")

In [0]:
display(
    spark.table("insureflow.audit.dq_results")
)

In [0]:
# if not spark.catalog.tableExists("insureflow.audit.dq_results"):

#     dq_result_df.write \
#         .format("delta") \
#         .mode("overwrite") \
#         .saveAsTable("insureflow.audit.dq_results")

# else:

#     dq_result_df.write \
#         .format("delta") \
#         .mode("append") \
#         .saveAsTable("insureflow.audit.dq_results")

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

dq_schema = StructType([
    StructField("rule_id", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("column_name", StringType(), False),
    StructField("rule_type", StringType(), False),
    StructField("rule_expression", StringType(), True),
    StructField("severity", StringType(), False),
    StructField("is_active", IntegerType(), False)
])

dq_rules_data = [
    (
        "DQ001",
        "policy",
        "policy_id",
        "NOT_NULL",
        "policy_id IS NOT NULL",
        "CRITICAL",
        1
    ),
    (
        "DQ002",
        "policy",
        "policy_id",
        "DUPLICATE",
        "policy_id",
        "CRITICAL",
        1
    ),
    (
        "DQ003",
        "policy",
        "premium",
        "RANGE",
        "premium >= 0",
        "HIGH",
        1
    ),
    (
        "DQ004",
        "policy",
        "status",
        "ALLOWED_VALUE",
        "Pending,Expired,Active,Cancelled",
        "HIGH",
        1
    ),
    (
        "DQ005",
        "policy",
        "effective_date,expiry_date",
        "DATE_ORDER",
        "effective_date <= expiry_date",
        "HIGH",
        1
    )
]

dq_rules_df = spark.createDataFrame(
    dq_rules_data,
    dq_schema
)

display(dq_rules_df)

In [0]:
dq_rules_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("insureflow.metadata.dq_rules")

In [0]:
display(
    spark.table("insureflow.metadata.dq_rules")
)

In [0]:
rules_df = spark.table("insureflow.metadata.dq_rules")

In [0]:
active_rules_df = rules_df.filter(
    F.col("is_active") == 1
)

display(active_rules_df)

In [0]:
rules = active_rules_df.collect()

for rule in rules:
    print(
        rule.rule_id,
        rule.rule_type,
        rule.column_name,
        rule.severity
    )

In [0]:
def execute_dq_rule(df, rule):

    rule_type = rule.rule_type
    column_name = rule.column_name

    parameters = parse_rule_parameters(
        rule.rule_parameters
    )

    if rule_type == "NOT_NULL":

        failed_records = df.filter(
            F.col(column_name).isNull()
        ).count()

    elif rule_type == "DUPLICATE":

        failed_records = (
            df.groupBy(column_name)
              .count()
              .filter(F.col("count") > 1)
              .count()
        )

    elif rule_type == "RANGE":

        min_value = float(
            parameters["min_value"]
        )

        failed_records = df.filter(
            F.col(column_name) < min_value
        ).count()

    elif rule_type == "ALLOWED_VALUE":

        allowed_values = parameters[
            "values"
        ].split(",")

        failed_records = df.filter(
            ~F.col(column_name).isin(allowed_values)
        ).count()

    elif rule_type == "DATE_ORDER":

        start_column = parameters[
            "start_column"
        ]

        end_column = parameters[
            "end_column"
        ]

        failed_records = df.filter(
            F.col(start_column) > F.col(end_column)
        ).count()

    else:

        raise ValueError(
            f"Unsupported rule type: {rule_type}"
        )

    total_records = df.count()

    return create_dq_result(
        rule_id=rule.rule_id,
        rule_type=rule.rule_type,
        rule_description=rule.rule_description,
        total_records=total_records,
        failed_records=failed_records,
        severity=rule.severity
    )

In [0]:
dq_results = []

for rule in rules:
    
    result = execute_dq_rule(
        df,
        rule
    )
    
    dq_results.append(result)

In [0]:
dq_result_df = spark.createDataFrame(dq_results)

display(dq_result_df)

In [0]:
display(
    spark.table("insureflow.metadata.dq_rules")
)

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

dq_schema_v2 = StructType([
    StructField("rule_id", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("column_name", StringType(), False),
    StructField("rule_type", StringType(), False),
    StructField("rule_parameters", StringType(), True),
    StructField("rule_description", StringType(), True),
    StructField("severity", StringType(), False),
    StructField("is_active", IntegerType(), False)
])

In [0]:
dq_rules_data_v2 = [

    (
        "DQ001",
        "policy",
        "policy_id",
        "NOT_NULL",
        None,
        "policy_id must not be NULL",
        "CRITICAL",
        1
    ),

    (
        "DQ002",
        "policy",
        "policy_id",
        "DUPLICATE",
        None,
        "policy_id must be unique",
        "CRITICAL",
        1
    ),

    (
        "DQ003",
        "policy",
        "premium",
        "RANGE",
        "min_value=0",
        "premium must be greater than or equal to zero",
        "HIGH",
        1
    ),

    (
        "DQ004",
        "policy",
        "status",
        "ALLOWED_VALUE",
        "values=Pending|Expired|Active|Cancelled",
        "status must contain an allowed value",
        "HIGH",
        1
    ),

    (
        "DQ005",
        "policy",
        "effective_date,expiry_date",
        "DATE_ORDER",
        "start_column=effective_date|end_column=expiry_date",
        "effective_date must be less than or equal to expiry_date",
        "HIGH",
        1
    )
]

In [0]:
dq_rules_v2_df = spark.createDataFrame(
    dq_rules_data_v2,
    dq_schema_v2
)

display(dq_rules_v2_df)

In [0]:
dq_rules_v2_df = dq_rules_v2_df.withColumn(
    "rule_parameters",
    F.when(
        F.col("rule_id") == "DQ004",
        F.lit("values=Pending,Expired,Active,Cancelled")
    ).otherwise(F.col("rule_parameters"))
)

In [0]:
def parse_rule_parameters(parameter_string):

    if parameter_string is None:
        return {}

    parameter_string = parameter_string.strip()

    if parameter_string == "":
        return {}

    parameters = {}

    for item in parameter_string.split("|"):

        item = item.strip()

        if "=" not in item:
            continue

        key, value = item.split("=", 1)

        parameters[key.strip()] = value.strip()

    return parameters

In [0]:
print(
    parse_rule_parameters("min_value=0")
)

print(
    parse_rule_parameters(
        "start_column=effective_date|end_column=expiry_date"
    )
)

In [0]:
rules = (
    dq_rules_v2_df
    .filter(F.col("is_active") == 1)
    .collect()
)

dq_results = []

for rule in rules:

    result = execute_dq_rule(
        df,
        rule
    )

    dq_results.append(result)

dq_result_df = spark.createDataFrame(dq_results)

display(dq_result_df)

In [0]:
display(
    df.groupBy("status")
      .count()
      .orderBy(F.desc("count"))
)

In [0]:
dq_rules_v2_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("insureflow.metadata.dq_rules")

In [0]:
display(
    spark.table("insureflow.metadata.dq_rules")
)

In [0]:
import uuid
from pyspark.sql import functions as F


def run_dq(df, table_name):

    # 1. Generate unique run ID
    run_id = str(uuid.uuid4())

    # 2. Load active DQ rules
    rules_df = (
        spark.table("insureflow.metadata.dq_rules")
        .filter(F.col("is_active") == 1)
    )

    rules = rules_df.collect()

    # 3. Execute rules
    dq_results = []

    for rule in rules:

        result = execute_dq_rule(
            df,
            rule
        )

        dq_results.append(result)

    # 4. Create result DataFrame
    dq_result_df = spark.createDataFrame(
        dq_results
    )

    # 5. Add execution metadata
    dq_result_df = (
        dq_result_df
        .withColumn(
            "run_id",
            F.lit(run_id)
        )
        .withColumn(
            "table_name",
            F.lit(table_name)
        )
        .withColumn(
            "execution_timestamp",
            F.current_timestamp()
        )
    )

    # 6. Calculate overall status
    failed_rules = dq_result_df.filter(
        F.col("status") == "FAIL"
    ).count()

    overall_status = (
        "PASS"
        if failed_rules == 0
        else "FAIL"
    )

    # 7. Write audit results
    dq_result_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(
            "insureflow.audit.dq_results"
        )

    # 8. Return results + overall status
    return dq_result_df, overall_status

In [0]:
dq_result_df, overall_status = run_dq(
    df,
    "policy"
)

In [0]:
display(dq_result_df)

print(
    "Overall DQ Status:",
    overall_status
)

In [0]:
run_dq(df, "policy")

In [0]:
display(
    spark.table("insureflow.audit.dq_results")
    .orderBy(
        F.col("execution_timestamp").desc()
    )
)

In [0]:
test_df = (
    df.withColumn(
        "premium",
        F.when(
            F.monotonically_increasing_id() == 0,
            F.lit(-500)
        ).otherwise(F.col("premium"))
    )
)

In [0]:
test_result_df, test_status = run_dq(
    test_df,
    "policy_test"
)

In [0]:
display(test_result_df)

print(
    "Overall DQ Status:",
    test_status
)

In [0]:
failed_dq003_df = test_df.filter(
    F.col("premium") < 0
)

display(failed_dq003_df)

In [0]:
run_id = str(uuid.uuid4())

dq_failed_df = (
    failed_dq003_df
    .withColumn("run_id", F.lit(run_id))
    .withColumn("rule_id", F.lit("DQ003"))
    .withColumn("rule_type", F.lit("RANGE"))
    .withColumn("severity", F.lit("HIGH"))
    .withColumn(
        "failure_reason",
        F.lit("premium must be greater than or equal to zero")
    )
    .withColumn("failure_timestamp", F.current_timestamp())
)

display(dq_failed_df)

In [0]:
if not spark.catalog.tableExists("insureflow.audit.dq_failed_records"):

    dq_failed_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("insureflow.audit.dq_failed_records")

else:
    dq_failed_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("insureflow.audit.dq_failed_records")

In [0]:
display(
    spark.table("insureflow.audit.dq_failed_records")
)

In [0]:
from pyspark.sql import functions as F

def get_failed_records(df, rule):

    rule_type = rule.rule_type
    column_name = rule.column_name
    parameters = parse_rule_parameters(rule.rule_parameters)

    if rule_type == "NOT_NULL":

        failed_df = df.filter(
            F.col(column_name).isNull()
        )

    elif rule_type == "DUPLICATE":

        duplicate_keys = (
            df.groupBy(column_name)
              .count()
              .filter(F.col("count") > 1)
              .select(column_name)
        )

        failed_df = (
            df.join(
                duplicate_keys,
                on=column_name,
                how="inner"
            )
        )

    elif rule_type == "RANGE":

        min_value = float(parameters["min_value"])

        failed_df = df.filter(
            F.col(column_name) < min_value
        )

    elif rule_type == "ALLOWED_VALUE":

        allowed_values = parameters["values"].split(",")

        failed_df = df.filter(
            ~F.col(column_name).isin(allowed_values)
        )

    elif rule_type == "DATE_ORDER":

        start_column = parameters["start_column"]
        end_column = parameters["end_column"]

        failed_df = df.filter(
            F.col(start_column) > F.col(end_column)
        )

    else:
        raise ValueError(
            f"Unsupported rule type: {rule_type}"
        )

    return failed_df

In [0]:
rule_dq003 = (
    spark.table("insureflow.metadata.dq_rules")
    .filter(F.col("rule_id") == "DQ003")
    .first()
)

In [0]:
failed_df = get_failed_records(
    test_df,
    rule_dq003
)

display(failed_df)

In [0]:
run_id = str(uuid.uuid4())

failed_dq003_with_metadata = (
    failed_df
    .withColumn("run_id", F.lit(run_id))
    .withColumn("rule_id", F.lit(rule_dq003.rule_id))
    .withColumn("rule_type", F.lit(rule_dq003.rule_type))
    .withColumn("severity", F.lit(rule_dq003.severity))
    .withColumn(
        "failure_reason",
        F.lit(rule_dq003.rule_description)
    )
    .withColumn(
        "failure_timestamp",
        F.current_timestamp()
    )
)

display(failed_dq003_with_metadata)

In [0]:
def run_dq(df, table_name):

    # 1. Generate unique run ID
    run_id = str(uuid.uuid4())

    # 2. Load active DQ rules
    rules_df = (
        spark.table("insureflow.metadata.dq_rules")
        .filter(F.col("is_active") == 1)
    )

    rules = rules_df.collect()

    # Lists to store results
    dq_results = []
    failed_record_dfs = []

    # 3. Execute each DQ rule
    for rule in rules:

        # Get summary result
        result = execute_dq_rule(df, rule)
        dq_results.append(result)

        # Get actual failed records
        failed_df = get_failed_records(df, rule)

        # Only process if failures exist
        if failed_df.limit(1).count() > 0:

            failed_df = (
                failed_df
                .withColumn("run_id", F.lit(run_id))
                .withColumn("rule_id", F.lit(rule.rule_id))
                .withColumn("rule_type", F.lit(rule.rule_type))
                .withColumn("severity", F.lit(rule.severity))
                .withColumn(
                    "failure_reason",
                    F.lit(rule.rule_description)
                )
                .withColumn(
                    "failure_timestamp",
                    F.current_timestamp()
                )
            )

            failed_record_dfs.append(failed_df)

    # 4. Create DQ summary DataFrame
    dq_result_df = spark.createDataFrame(dq_results)

    dq_result_df = (
        dq_result_df
        .withColumn("run_id", F.lit(run_id))
        .withColumn("table_name", F.lit(table_name))
        .withColumn(
            "execution_timestamp",
            F.current_timestamp()
        )
    )

    # 5. Calculate overall status
    failed_rules = (
        dq_result_df
        .filter(F.col("status") == "FAIL")
        .count()
    )

    overall_status = (
        "PASS"
        if failed_rules == 0
        else "FAIL"
    )

    # 6. Write DQ summary
    dq_result_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("insureflow.audit.dq_results")

    # 7. Write failed records
    if failed_record_dfs:

        all_failed_records_df = failed_record_dfs[0]

        for failed_df in failed_record_dfs[1:]:
            all_failed_records_df = (
                all_failed_records_df.unionByName(
                    failed_df
                )
            )

        # Add table name
        all_failed_records_df = (
            all_failed_records_df
            .withColumn(
                "table_name",
                F.lit(table_name)
            )
        )

        all_failed_records_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(
                "insureflow.audit.dq_failed_records"
            )

    # 8. Return results
    return dq_result_df, overall_status

In [0]:
print(
    spark.catalog.tableExists(
        "insureflow.audit.dq_failed_records"
    )
)

In [0]:
test_result_df, test_status = run_dq(
    test_df,
    "policy_test"
)

display(test_result_df)

print("Overall Status:", test_status)

In [0]:
display(
    spark.table(
        "insureflow.audit.dq_failed_records"
    )
)